<a href="https://colab.research.google.com/github/rebirthmagex/Jogo_Bagha_Chall/blob/main/engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from contextlib import suppress
from pprint import pprint

import math
from enum import Enum
import copy
from copy import deepcopy
from collections import deque

import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
!pip install import-ipynb
import import_ipynb

In [ ]:
from board import Board

In [ ]:
class Engine:
    def __init__(self, selected_goat_heuristics, selected_tiger_heuristics, depth=5):
        self.board = Board()  # Inicializa uma instância de `Board`
        self.depth = depth
        self.best_move = None
        self.selected_goat_heuristics = selected_goat_heuristics
        self.selected_tiger_heuristics = selected_tiger_heuristics

    def apply_goat_heuristics(self):
        base_value = 0
        # print("\nHeurísticas Selecionadas para Cabras:")
        for heuristic in self.selected_goat_heuristics:
              # Aplicando as heurísticas selecionadas para Cabras
              if heuristic == "Maximizar a Oportunidade de Movimentos":
                  base_value += -50 * self.board.get_amount_of_goat_moves(self.board.game_board)
              if heuristic == "Minimizar Capturas":
                  base_value += -1000 * self.board.minimize_captures(self.board.game_board)
              if heuristic == "Maximizar Proteção":
                  base_value += -500 * self.board.maximize_protection(self.board.game_board)
              if heuristic == "Posicionar no Centro":
                  base_value += -400 * self.board.position_in_center(self.board.game_board)
              if heuristic == "Evitar Diagonais":
                  base_value += 200 * self.board.avoid_diagonals(self.board.game_board)
              if heuristic == "Maximixar a Oportunidade de Espaços Fechados":
                  base_value += -700 * self.board.no_of_closed_spaces(self.board.game_board)
        return base_value

    def apply_tiger_heuristics(self):
        base_value = 0
        # print("\nHeurísticas Selecionadas para Tigres:")
        for heuristic in self.selected_tiger_heuristics:
              # Aplicando as heurísticas selecionadas para Tigres
              if heuristic == "Maximizar a Oportunidade de Movimentos":
                  base_value += 50 * self.board.get_amount_of_tiger_moves(self.board.game_board)
              if heuristic == "Maximizar Capturas":
                  base_value += 1000 * self.board.tiger_potential_captures(self.board.game_board)
              if heuristic == "Ocupar Linhas Importantes":
                  base_value += 200 * self.board.occupy_important_lines(self.board.game_board)
              if heuristic == "Evitar Encurralamento":
                  base_value += -1000 * self.board.avoid_trapping(self.board.game_board)
              if heuristic == "Atacar em Grupo":
                  base_value -= 300 * self.board.group_attack(self.board.game_board)
              if heuristic == "Maximizar a Morte das Cabras":
                  base_value -= 1000 * self.board.dead_goats
              if heuristic == "Mobilidade dps Tigres":
                  base_value -= 200 * self.board.count_movable_tigers(self.board.game_board)
        return base_value

    def evaluate(self, depth=0):
        """ Retorna uma avaliação numérica da posição, considerando as heurísticas selecionadas. """
        winner = self.board.winner
        if not winner:
            return self.apply_goat_heuristics() + self.apply_tiger_heuristics()
        return self.evaluate_winner(winner)

    def evaluate_winner(self, winner):
        if winner == self.board.Player.GOAT:
            return -math.inf
        elif winner == self.board.Player.TIGER:
            return math.inf

    def minmax(self, is_max=True, depth=0, alpha=-math.inf, beta=math.inf):
        """ Implementa o algoritmo Minimax com poda Alfa-Beta. """
        score = self.evaluate(depth)
        if depth == self.depth or abs(score) == math.inf:
            return score

        return self.min_or_max(is_max, depth, alpha, beta)

    def min_or_max(self, is_max, depth, alpha, beta):
        value = math.inf if not is_max else -math.inf
        moves = self.board.generate_move_list(self.board.game_board)

        for move in moves:
            self.board.make_move(move)
            value_t = self.minmax(not is_max, depth + 1, alpha, beta)
            self.board.revert_move(move)

            if is_max:
                value, alpha = self.maximize(value, value_t, alpha, move, depth)
            else:
                value, beta = self.minimize(value, value_t, beta, move, depth)

            if alpha >= beta:
                break

        return value

    def maximize(self, value, value_t, alpha, move, depth):
        if value_t > value:
            value = value_t
            alpha = max(alpha, value)
            if depth == 0:
                self.best_move = move
        return value, alpha

    def minimize(self, value, value_t, beta, move, depth):
        if value_t < value:
            value = value_t
            beta = min(beta, value)
            if depth == 0:
                self.best_move = move
        return value, beta

    def get_best_move(self):
        if self.board.turn == self.board.Player.GOAT:
            self.minmax(is_max=False)
        else:
            self.minmax()
        return self.best_move

    def make_best_move(self):
        move = self.get_best_move()
        self.board.make_move(move)